# EDA - AnimeList Management

In [ ]:
import glob
import os

import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
plt.rcParams["figure.figsize"] = (10, 5)

## 1. Caricamento dati

Trova e carica automaticamente il file più recente in `data/finito/`.

In [ ]:
FINITO_DATA_PATH = "../data/finito"

csv_files = sorted(glob.glob(os.path.join(FINITO_DATA_PATH, "anime_finito_*.csv")))

if not csv_files:
    raise FileNotFoundError(
        f"Nessun file trovato in {FINITO_DATA_PATH}. "
        "Lancia prima la pipeline con 'python main.py' per generarne uno."
    )

latest_file = csv_files[-1]
print(f"Carico: {latest_file}")

df = pd.read_csv(latest_file, parse_dates=["aired_from", "aired_to"])
print(f"Record caricati: {len(df)}")
df.head()

## 2. Panoramica generale

In [ ]:
df.info()

In [ ]:
df.describe(include="number")

## 3. Valori mancanti per colonna


In [ ]:
missing = df.isna().sum().sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(1)
pd.DataFrame({"mancanti": missing, "percentuale_%": missing_pct})

## 4. Distribuzione dei punteggi (score)

In [ ]:
df["score"].dropna().hist(bins=30)
plt.title("Distribuzione degli score")
plt.xlabel("Score")
plt.ylabel("Numero di anime")
plt.show()

## 5. Top 10 anime per punteggio

In [ ]:
top_score = (
    df[df["scored_by"] >= 1000]
    .sort_values("score", ascending=False)
    .loc[:, ["title", "score", "scored_by", "members", "type_"]]
    .head(10)
)
top_score

## 6. Top 10 anime per popolarità (membri)

In [ ]:
top_members = (
    df.sort_values("members", ascending=False)
    .loc[:, ["title", "members", "score", "type_"]]
    .head(10)
)
top_members

## 7. Anime per anno e stagione

In [ ]:
anime_per_anno = df.dropna(subset=["year_"]).groupby("year_").size()
anime_per_anno.tail(20).plot(kind="bar")
plt.title("Anime pubblicati per anno (ultimi 20 anni)")
plt.xlabel("Anno")
plt.ylabel("Numero di anime")
plt.show()

In [ ]:
df["season"].value_counts(dropna=True).plot(kind="bar", color="orange")
plt.title("Distribuzione per stagione")
plt.xlabel("Stagione")
plt.ylabel("Numero di anime")
plt.show()

## 8. Tipo di anime (TV, Movie, OVA, ecc.)

In [ ]:
df["type_"].value_counts().plot(kind="pie", autopct="%1.1f%%")
plt.title("Distribuzione per tipo")
plt.ylabel("")
plt.show()

## 9. Generi più frequenti

In [ ]:
generi = (
    df["genres"]
    .dropna()
    .str.split(", ")
    .explode()
    .str.strip()
    .value_counts()
)

generi.head(15).plot(kind="barh")
plt.title("Top 15 generi più frequenti")
plt.xlabel("Numero di anime")
plt.gca().invert_yaxis()
plt.show()

## 10. Studio più prolifici

In [ ]:
studi = (
    df["studios"]
    .dropna()
    .str.split(", ")
    .explode()
    .str.strip()
    .value_counts()
)

studi.head(15).plot(kind="barh", color="green")
plt.title("Top 15 studi per numero di anime prodotti")
plt.xlabel("Numero di anime")
plt.gca().invert_yaxis()
plt.show()

## 11. Relazione tra popolarità e punteggio

In [ ]:
plt.scatter(df["members"], df["score"], alpha=0.3, s=10)
plt.xscale("log")
plt.xlabel("Membri (scala log)")
plt.ylabel("Score")
plt.title("Popolarità vs Punteggio")
plt.show()